# Football Detection - `best.pt` Test Notebook

Bu notebook kendi videonla `best.pt` modelini Colab üzerinde test etmek için hazırlandı.

Akış:
1. GPU kontrolü
2. Gerekli paketleri kurma
3. `best.pt` ve video dosyasını Colab dizinine yükleme
4. YOLO ile inference alma
5. Annotated çıktı videosunu üretme
6. Sınıf bazlı tespit sayılarını özetleme
7. Çıktı videosunu indirme

> `best.pt` dosyasını doğrudan Colab ana dizinine veya upload hücresiyle yükleyebilirsin.

In [ ]:
# 1) GPU kontrolü
!nvidia-smi

In [ ]:
# 2) Gerekli paketleri kur
!pip install -q ultralytics opencv-python-headless supervision pandas tqdm

import os
import cv2
import glob
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode

print("Kurulum tamam.")

## 3) `best.pt` ve video dosyasını yükle

Aşağıdaki hücreyi çalıştırınca dosya seçme penceresi açılır. Buradan:
- `best.pt`
- test edeceğin video dosyası, örneğin `mac.mp4`

yükleyebilirsin.

Eğer dosyaları Colab sol panelinden elle yüklediysen bu hücreyi çalıştırmana gerek yok; sonraki hücre dosyaları bulmaya çalışır.

In [ ]:
# Dosya yükleme: best.pt ve video dosyanı seç
uploaded = files.upload()
print("Yüklenen dosyalar:", list(uploaded.keys()))

In [ ]:
# 4) Model ve video yolunu otomatik bul
MODEL_PATH = "best.pt"

video_extensions = ["*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm"]
video_files = []
for ext in video_extensions:
    video_files.extend(glob.glob(ext))

if not os.path.exists(MODEL_PATH):
    pt_files = glob.glob("*.pt")
    if len(pt_files) == 0:
        raise FileNotFoundError("Colab dizininde .pt model dosyası bulunamadı. Lütfen best.pt yükle.")
    MODEL_PATH = pt_files[0]

if len(video_files) == 0:
    raise FileNotFoundError("Colab dizininde video bulunamadı. Lütfen .mp4/.avi/.mov/.mkv/.webm video yükle.")

VIDEO_PATH = video_files[0]

print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)

## 5) Hızlı YOLO testi

Bu hücre videoda modeli test eder ve Ultralytics'in kendi anotasyonlu çıktısını üretir. Önce bunu çalıştırmak hızlı kontrol için yeterli.

In [ ]:
# Hızlı test ayarları
CONF = 0.15      # top gibi küçük nesneler için düşük tutuldu
IOU = 0.45
IMGSZ = 1280     # küçük top için 640 yerine 1280 daha iyi olabilir
MAX_DET = 300

model = YOLO(MODEL_PATH)
print("Model sınıfları:", model.names)

results = model.predict(
    source=VIDEO_PATH,
    conf=CONF,
    iou=IOU,
    imgsz=IMGSZ,
    max_det=MAX_DET,
    save=True,
    project="runs/detect",
    name="football_test",
    exist_ok=True,
    verbose=True
)

print("Hızlı test tamamlandı. Çıktı klasörü: runs/detect/football_test")

In [ ]:
# Ultralytics'in ürettiği çıktı videoyu bul
output_candidates = []
for ext in video_extensions:
    output_candidates.extend(glob.glob(f"runs/detect/football_test/{ext}"))

print("Bulunan çıktılar:", output_candidates)

if output_candidates:
    YOLO_OUTPUT_VIDEO = output_candidates[0]
    print("Çıktı videosu:", YOLO_OUTPUT_VIDEO)
else:
    YOLO_OUTPUT_VIDEO = None
    print("Çıktı videosu bulunamadı. Bir sonraki özel çizim hücresini kullanabilirsin.")

## 6) Özel çıktı üret: bbox, label, confidence ve sınıf sayımı

Bu hücre frame frame videoyu işler, anotasyonlu yeni video üretir ve kaç adet `player / ball / referee / goalkeeper` görüldüğünü özetler.

In [ ]:
# Özel anotasyonlu video üretimi
OUTPUT_VIDEO = "football_bestpt_test_output.mp4"
OUTPUT_CSV = "football_bestpt_detection_summary.csv"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Video açılamadı: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

records = []
frame_idx = 0

pbar = tqdm(total=frame_count, desc="Video işleniyor")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.predict(
        source=frame,
        conf=CONF,
        iou=IOU,
        imgsz=IMGSZ,
        max_det=MAX_DET,
        verbose=False
    )[0]

    annotated = frame.copy()

    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int).tolist()
            label = model.names.get(cls_id, str(cls_id))

            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            text = f"{label} {conf:.2f}"
            cv2.putText(
                annotated,
                text,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )

            records.append({
                "frame": frame_idx,
                "class_id": cls_id,
                "class_name": label,
                "confidence": conf,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": int((x1 + x2) / 2),
                "center_y": int((y1 + y2) / 2),
            })

    out.write(annotated)
    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
out.release()

summary_df = pd.DataFrame(records)
summary_df.to_csv(OUTPUT_CSV, index=False)

print("Özel çıktı videosu:", OUTPUT_VIDEO)
print("CSV çıktı:", OUTPUT_CSV)
print("Toplam detection:", len(summary_df))

if len(summary_df) > 0:
    display(summary_df["class_name"].value_counts().rename_axis("class").reset_index(name="count"))
    display(summary_df.head())
else:
    print("Hiç detection bulunamadı. CONF değerini düşürmeyi deneyebilirsin: örn. CONF = 0.05")

## 7) Çıktı videosunu notebook içinde önizle

Video büyükse Colab önizleme yavaş olabilir. İndirme hücresi daha güvenilir çalışır.

In [ ]:
# Notebook içinde video önizleme
preview_path = OUTPUT_VIDEO

if os.path.exists(preview_path):
    mp4 = open(preview_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    html = """
    <video width="900" controls>
        <source src="{}" type="video/mp4">
    </video>
    """.format(data_url)
    display(HTML(html))
else:
    print("Önizlenecek video bulunamadı.")

## 8) Çıktıları indir

Aşağıdaki hücre anotasyonlu videoyu ve detection CSV dosyasını indirir.

In [ ]:
# Çıktıları indir
if os.path.exists(OUTPUT_VIDEO):
    files.download(OUTPUT_VIDEO)

if os.path.exists(OUTPUT_CSV):
    files.download(OUTPUT_CSV)

## 9) Ayar önerileri

Top veya uzaktaki oyuncular kaçıyorsa yukarıdaki ayarları değiştir:

```python
CONF = 0.05
IMGSZ = 1280 veya 1536
IOU = 0.45
```

Video çok yavaş işleniyorsa:

```python
IMGSZ = 640 veya 960
CONF = 0.20
```

Sonraki aşamada bu notebook'a ByteTrack, takım rengi ayrımı, topa sahip olma ve oyuncu istatistikleri eklenebilir.